In [ ]:
import os
from dotenv import load_dotenv
from langfuse import get_client
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from smolagents import CodeAgent, LiteLLMModel, GoogleSearchTool, VisitWebpageTool

load_dotenv(override=True)

# Tracing
langfuse = get_client()
print("Langfuse ready:", langfuse.auth_check())
SmolagentsInstrumentor().instrument()

# Shared model for both agents
model = LiteLLMModel(
    model_id="gemini/gemini-3.5-flash-lite",
    api_key=os.environ["GEMINI_API_KEY"],
    timeout=60,
    num_retries=0,
)

model2 = LiteLLMModel(
    model_id="ollama_chat/qwen2.5-coder:14b",
    api_base="http://localhost:11434",
    num_ctx=16384,
)

/home/lucas/Documents/ia/hf-agents-course/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Langfuse ready: True


Initialize the sub agent who browse and retrieve information from internet websites

In [2]:
# Sub-agent: only browses the web, no extra imports
web_agent = CodeAgent(
    model=model,
    tools=[
        GoogleSearchTool(provider="serper"),
        VisitWebpageTool(max_output_length=5000),
        ],
    name="web_agent",
    description=(
        "Searches the web and reads webpages."
        "Use it for any information that must be found online."
        "Give it a precise, self-contained question: it has no access to the manager's context."
    ),
    max_steps=4,
)

Initialize the manager of the sub-agent

In [3]:
# Manager: delegates web lookups, never browses itself
manager_agent = CodeAgent(
    model=model,
    tools=[],
    managed_agents=[web_agent],
    max_steps=3,
    # Sub-agent calls run inside the manager's code step: allow time for a full web_agent run
    executor_kwargs={"timeout_seconds": 600},
)

# Check the multi-agent setup
manager_agent.visualize()

CodeAgent | gemini/gemini-3.5-flash
├── ✅ Authorized imports: []
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name         ┃ Description                                   ┃ Arguments                                    ┃
│   ┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ final_answer │ Provides a final answer to the given problem. │ answer (`any`): The final answer to the      │
│   │              │                                               │ problem                                      │
│   └──────────────┴───────────────────────────────────────────────┴──────────────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | gemini/gemini-3.5-flash
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Searches the web and reads webpages.Use it for any information that must be found 
        │   online.Give it a precise, self-contained question: it has no access to the manager's context.
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name          ┃ Description                              ┃ Arguments                                ┃
            ┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search    │ Performs a google web search for your    │ query (`string`): The search query to    │
            │               │ query then returns a string of the top   │ perform.                                 │
            │               │ search results.                          │ filter_year (`integer`): Optionally      │
            │               │                                          │ restrict results to a certain year       │
            │ visit_webpage │ Visits a webpage at the given url and    │ url (`string`): The url of the webpage   │
            │               │ reads its content as a markdown string.  │ to visit.                                │
            │               │ Use this to browse webpages.             │                                          │
            │ final_answer  │ Provides a final answer to the given     │ answer (`any`): The final answer to the  │
            │               │ problem.                                 │ problem                                  │
            └───────────────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

In [4]:
result = manager_agent.run(
    "Find 3 private banks headquartered in Geneva and give the official website of each."
)
print(result)

langfuse.flush()

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find 3 private banks headquartered in Geneva and give the official website of each.                             │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.5-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.



Error in generating model output:
litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please 
try again later.",
    "status": "UNAVAILABLE"
  }
}

[Step 1: Duration 14.85 seconds]

AgentGenerationError: Error in generating model output:
litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",
    "status": "UNAVAILABLE"
  }
}
